In [ ]:
from pyspark.sql import SparkSession
import os
import os
import sys

# Go to project root
project_root = os.path.abspath("/home/nhatquang/home-credit-credit-risk-model-stability")
os.chdir(project_root)
sys.path.insert(0, project_root)  # optional: to import modules from root

# --- Configurations ---
service_account_key = os.path.abspath("secrets/global-phalanx-449403-d2-aae80316b9df.json")
gcs_connector_path = os.path.abspath("data-platform/jars/gcs-connector-hadoop3-2.2.9-shaded.jar")

# Set environment variable (if not already set)
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 6g --executor-memory 6g pyspark-shell"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = service_account_key
os.environ["PYSPARK_SUBMIT_ARGS"] = "--jars data-platform/jars/gcs-connector-hadoop3-2.2.9-shaded.jar pyspark-shell"
# Create Spark session with GCS configs
# Remove the 2 last config before push to production
spark = SparkSession.builder \
    .appName("ReadGCSCSV") \
    .config("spark.jars", gcs_connector_path) \
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS") \
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .config("spark.hadoop.google.cloud.auth.service.account.json.keyfile", service_account_key) \
    .config("spark.sql.shuffle.partitions", "1") \
    .config("spark.default.parallelism", "1") \
    .getOrCreate()

25/07/28 18:13:12 WARN Utils: Your hostname, laboratory resolves to a loopback address: 127.0.1.1; using 192.168.1.42 instead (on interface enx00e04c3600c1)
25/07/28 18:13:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/07/28 18:13:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
from pyspark.sql import SparkSession

# Path to your CSV in GCS
gcs_data_path = "gs://credit-risk-modeling-bucket/raw_data/"
gcs_train_file = "application_train.csv"
gcs_bureau_file = "bureau.csv"
gcs_bureau_balance_file = "bureau_balance.csv"
gcs_credit_card_balance = "credit_card_balance.csv"
gcs_installments_payments = "installments_payments.csv"
gcs_previous_application = "previous_application.csv"
gcs_pos_cash_balance = "POS_CASH_balance.csv"
gcs_connector_jars = "gs://credit-risk-modeling-bucket/jars/gcs-connector-hadoop3-latest.jar "

# spark = SparkSession.builder \
#         .master("local[*]") \
#         .appName("CreditRiskDataLoading") \
#         .config("spark.jars", gcs_connector_jars) \
#         .getOrCreate()

def load_csv_to_df(file_path, df_name): 
    """
    Helper function to load a CSV file into a Spark DataFrame and print info.
    """
    try:
        df = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(file_path) \
            .sample(fraction=0.1, seed=42)
        print(f"✅ DataFrame '{df_name}' loaded successfully from GCS using Spark!")
        return df
    except Exception as e:
        print(f"❌ Error reading '{df_name}' from GCS: {e}")
        raise 
# Load all DataFrames
print("--- Loading application_train.csv ---")
train_df = load_csv_to_df(gcs_data_path + gcs_train_file, "train_df")

print("\n--- Loading bureau.csv ---")
bureau_df = load_csv_to_df(gcs_data_path + gcs_bureau_file, "bureau_df")

print("\n--- Loading bureau_balance.csv ---")
bureau_balance_df = load_csv_to_df(gcs_data_path + gcs_bureau_balance_file, "bureau_balance_df")

print("\n--- Loading POS_CASH_balance.csv ---")
pos_cash_balance_df = load_csv_to_df(gcs_data_path + gcs_pos_cash_balance, "pos_cash_balance_df")

print("\n--- Loading credit_card_balance.csv ---")
credit_card_balance_df = load_csv_to_df(gcs_data_path + gcs_credit_card_balance, "credit_card_balance_df")

print("\n--- Loading installments_payments.csv ---")
installments_payments_df = load_csv_to_df(gcs_data_path + gcs_installments_payments, "installments_payments_df")

print("\n--- Loading previous_application.csv ---")
previous_application_df = load_csv_to_df(gcs_data_path + gcs_previous_application, "previous_application_df")


# spark.stop()

--- Loading application_train.csv ---


✅ DataFrame 'train_df' loaded successfully from GCS using Spark!

--- Loading bureau.csv ---


✅ DataFrame 'bureau_df' loaded successfully from GCS using Spark!

--- Loading bureau_balance.csv ---


✅ DataFrame 'bureau_balance_df' loaded successfully from GCS using Spark!

--- Loading POS_CASH_balance.csv ---


✅ DataFrame 'pos_cash_balance_df' loaded successfully from GCS using Spark!

--- Loading credit_card_balance.csv ---


✅ DataFrame 'credit_card_balance_df' loaded successfully from GCS using Spark!

--- Loading installments_payments.csv ---


✅ DataFrame 'installments_payments_df' loaded successfully from GCS using Spark!

--- Loading previous_application.csv ---


✅ DataFrame 'previous_application_df' loaded successfully from GCS using Spark!


# Joining Dataframe

In [4]:
from pyspark.sql.functions import col

def prefix_df(df, prefix):
    new_cols = [prefix + c for c in df.columns]
    return df.toDF(*new_cols)

t = prefix_df(train_df, "t_")
b = prefix_df(bureau_df, "b_")
bb = prefix_df(bureau_balance_df, "bb_")
pa = prefix_df(previous_application_df, "pa_")

# Now you can join safely using key t_SK_ID_CURR == b_SK_ID_CURR, etc.
application_df = t.join(b, col("t_SK_ID_CURR") == col("b_SK_ID_CURR"), "left") \
    .join(bb, col("b_SK_ID_BUREAU") == col("bb_SK_ID_BUREAU"), "left") \
    .join(pa, col("t_SK_ID_CURR") == col("pa_SK_ID_CURR"), "left")


# Try simple spark ml

In [5]:
from pyspark.sql.functions import col, when, isnan, count
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

In [6]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [7]:
application_dtypes = application_df.dtypes
cat_cols = [name for name, dtype in application_dtypes if dtype == "string"]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
# create one-hot encoders
encoders = [OneHotEncoder(inputCols=[f"{c}_idx"], outputCols=[f"{c}_vec"], dropLast=False) for c in cat_cols]
num_cols = [c for c in application_df.columns if c not in cat_cols + ['TARGET', 'SK_ID_CURR']]
assembler = VectorAssembler(
    inputCols=[f"{c}_vec" for c in cat_cols] + num_cols,
    outputCol="features"
)

In [10]:

dt = DecisionTreeClassifier(labelCol="t_TARGET", featuresCol="features")

pipeline = Pipeline(stages=indexers + encoders + [assembler, dt])

model = pipeline.fit(application_df)

# Cross-validation setup
# paramGrid = ParamGridBuilder() \
#     .addGrid(dt.maxDepth, [5, 10]) \
#     .addGrid(dt.minInstancesPerNode, [1, 5]) \
#     .build()

# cv = CrossValidator(
#     estimator=pipeline,
#     evaluator=BinaryClassificationEvaluator(labelCol="t_TARGET"),
#     estimatorParamMaps=paramGrid,
#     numFolds=3
# )

# cvModel = cv.fit(application_df)
# result = cvModel.transform(application_df)


25/07/28 19:54:58 ERROR Executor: Exception in task 0.0 in stage 2390.0 (TID 802)
org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] Failed to execute user defined function (`VectorAssembler$$Lambda$5103/0x00007b2ff5576488`: (struct<t_NAME_CONTRACT_TYPE_vec:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,t_CODE_GENDER_vec:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,t_FLAG_OWN_CAR_vec:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,t_FLAG_OWN_REALTY_vec:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,t_NAME_TYPE_SUITE_vec:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,t_NAME_INCOME_TYPE_vec:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,t_NAME_EDUCATION_TYPE_vec:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,t_NAME_FAMILY_STATUS_vec:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,t_NAME_HOUSING_TYPE_vec:

Py4JJavaError: An error occurred while calling o43637.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 2390.0 failed 1 times, most recent failure: Lost task 0.0 in stage 2390.0 (TID 802) (192.168.1.42 executor driver): org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] Failed to execute user defined function (`VectorAssembler$$Lambda$5103/0x00007b2ff5576488`: (struct<t_NAME_CONTRACT_TYPE_vec:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,t_CODE_GENDER_vec:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,t_FLAG_OWN_CAR_vec:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,t_FLAG_OWN_REALTY_vec:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,t_NAME_TYPE_SUITE_vec:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,t_NAME_INCOME_TYPE_vec:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,t_NAME_EDUCATION_TYPE_vec:struct<type

In [11]:
spark.stop()

In [12]:
application_df

DataFrame[t_SK_ID_CURR: int, t_TARGET: int, t_NAME_CONTRACT_TYPE: string, t_CODE_GENDER: string, t_FLAG_OWN_CAR: string, t_FLAG_OWN_REALTY: string, t_CNT_CHILDREN: int, t_AMT_INCOME_TOTAL: double, t_AMT_CREDIT: double, t_AMT_ANNUITY: double, t_AMT_GOODS_PRICE: double, t_NAME_TYPE_SUITE: string, t_NAME_INCOME_TYPE: string, t_NAME_EDUCATION_TYPE: string, t_NAME_FAMILY_STATUS: string, t_NAME_HOUSING_TYPE: string, t_REGION_POPULATION_RELATIVE: double, t_DAYS_BIRTH: int, t_DAYS_EMPLOYED: int, t_DAYS_REGISTRATION: double, t_DAYS_ID_PUBLISH: int, t_OWN_CAR_AGE: double, t_FLAG_MOBIL: int, t_FLAG_EMP_PHONE: int, t_FLAG_WORK_PHONE: int, t_FLAG_CONT_MOBILE: int, t_FLAG_PHONE: int, t_FLAG_EMAIL: int, t_OCCUPATION_TYPE: string, t_CNT_FAM_MEMBERS: double, t_REGION_RATING_CLIENT: int, t_REGION_RATING_CLIENT_W_CITY: int, t_WEEKDAY_APPR_PROCESS_START: string, t_HOUR_APPR_PROCESS_START: int, t_REG_REGION_NOT_LIVE_REGION: int, t_REG_REGION_NOT_WORK_REGION: int, t_LIVE_REGION_NOT_WORK_REGION: int, t_REG_C